In [0]:
import os
from typing import Optional
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.window import Window
import logging
from pyspark.sql import DataFrame
BASE_DIR = "/Volumes/data/default/cbs_data"

# 1. Initialize and configure the logger object
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("pyspark_logger")


In [0]:

def load_csvs_to_dict(base_dir):
    """
    Scans a directory for CSV files and loads them into a dictionary of DataFrames.
    
    Args:
        base_dir (str): The DBFS or Volume path to scan.
        
    Returns:
        dict: A dictionary where keys are filenames and values are Spark DataFrames.
    """
    df_dict = {}
    
    print(f"--- Accessing Path: {base_dir} ---")
    
    try:
        # List files using Databricks utilities
        files = dbutils.fs.ls(base_dir)
        csv_files = [f for f in files if f.name.endswith(".csv")]
        
        if not csv_files:
            print("Warning: No CSV files found in the specified directory.")
            return df_dict

        for f in csv_files:
            # Create a clean key (remove .csv extension)
            clean_name = f.name.replace(".csv", "")
            
            try:
                # Load the DataFrame
                df = (spark.read
                      .format("csv")
                      .option("header", "true")
                      .option("inferSchema", "true")
                      .load(f.path))
                
                df_dict[clean_name] = df
                print(f"✅ Loaded: '{clean_name}' [Rows: {df.count()}]")
                
            except Exception as e:
                print(f"❌ Failed to load '{f.name}': {e}")

    except Exception as e:
        print(f"Critical Error accessing directory: {e}")

    return df_dict
def Master_Data(BASE_DIR):
    dataframes = load_csvs_to_dict(BASE_DIR)
    if dataframes:
        dataframes["transactions"] = dataframes["transactions"].join(dataframes["accounts"], on="account_id", how="inner")
        dataframes["transactions"] = dataframes["transactions"].join(dataframes["customers"], on="customer_id", how="inner")
        Master_File = dataframes["transactions"]
        print(f"--- Master File: ---")
        print (f"✅ File Contains {Master_File.count()} rows")
    return Master_File




In [0]:
#(Task 1 A)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_customer_total_balance(BASE_DIR: str) -> DataFrame:
    ''' The aim of this function is to determine the total balance of each customer.
    The function should return a Spark DataFrame with three columns: customer_id, account_type and total_balance.
    The total_balance column should contain the sum of all account balances for each customer.
    '''
    try:
        # Assuming Master_Data is defined elsewhere and returns a Spark DataFrame
        master_df = Master_Data(BASE_DIR)
        
        # Aggregate the balances by customer and account type
        aggregated_df = (master_df
                         .groupBy("customer_id", "account_type")
                         .agg({"balance": "sum"})
                         .withColumnRenamed("sum(balance)", "total_balance"))
        
        # Select the final required columns
        final_df = aggregated_df.select(
            col("customer_id"), 
            col("account_type"), 
            col("total_balance")
        )
        
        return final_df

    except Exception as e:
        logger.error(f"An error occurred while calculating customer total balances: {str(e)}")
        raise e

total_balance_df = get_customer_total_balance(BASE_DIR)
display(total_balance_df)
    


In [0]:

#TASK 1B
def new_account_last_year() -> Optional[DataFrame]:
    """
    The aim of this function is to determine the number of new accounts opened in the last year.
    The function should return a Spark DataFrame with five columns: customer_id, first_name, last_name, 
    account_id and opening_date. The Max_Date points at the year 2024 therefore the opening date 
    should capture new accounts within this range starting 2023-09-17.
    """
    try:
        # Load the data (assuming Master_Data returns a DataFrame)
        Master_File: DataFrame = Master_Data(BASE_DIR)
        
        # 1. Get the Max Date (2024-09-17)
        # Using F.max is cleaner than the agg dict syntax
        max_date_row = Master_File.select(F.max("opening_date")).collect()
        
        if not max_date_row or max_date_row[0][0] is None:
            logger.info("No data found in Master_File or opening_date is empty.")
            return None
            
        Max_date = max_date_row[0][0]

        # 2. Calculate the Start Date (one year prior)
        # add_months handles leap years and different month lengths better than manual math
        start_date = F.add_months(F.lit(Max_date), -12)
        
        # 3. Filter and Select specific columns
        # We filter where opening_date is between 2023-09-17 and 2024-09-17
        result_df: DataFrame = Master_File.filter(
            (F.col("opening_date") > start_date) & 
            (F.col("opening_date") <= F.lit(Max_date))
        ).select(
            "customer_id", 
            "first_name", 
            "last_name", 
            "account_id", 
            "opening_date"
        )
        
        # Remove duplicates based on account_id
        result_df = result_df.dropDuplicates(["account_id"])
        
        # Optional: Count and show for debugging
        logger.info(f"Max Date: {Max_date}")
        logger.info(f"Filtering for accounts after: 2023-09-17")
        logger.info(f"Total new accounts: {result_df.count()}")
        
        return result_df

    except Exception as e:
        logger.error(f"An error occurred in new_account_last_year: {str(e)}", exc_info=True)
        return None

# Run
df_new_accounts: Optional[DataFrame] = new_account_last_year()
if df_new_accounts is not None:
    display(df_new_accounts)
else:
    print("No data returned due to an error or empty source.")


In [0]:
#TASK 1C
def TopN_customer_total_balance(BASE_DIR: str, n: int = 5) -> Optional[DataFrame]:
    """
    Get TopN customers with the highest total balance across all accounts.
    
    The function returns a Spark DataFrame with three columns: first_name, last_name, and total_balance.
    The total_balance column contains the sum of all account balances for each customer.
    """
    try:
        # Load the master data (assuming Master_Data returns a Spark DataFrame)
        Master_File: DataFrame = Master_Data(BASE_DIR)
        
        # Aggregate, rename, sort, and limit to top N customers
        # Using explicit F.sum() is preferred over the dictionary syntax for cleaner execution
        topn: DataFrame = (
            Master_File.groupBy("first_name", "last_name")
            .agg(F.sum("balance").alias("total_balance"))
            .orderBy(F.desc("total_balance"))
            .limit(n)
        )
        
        # Explicitly select the target columns using F.col to match the docstring exactly
        result_df: DataFrame = topn.select(
            F.col("first_name"), 
            F.col("last_name"), 
            F.col("total_balance")
        )
        
        # Log successful execution metrics
        logger.info(f"Successfully calculated top {n} customers by total balance.")
        
        return result_df

    except Exception as e:
        # Capture schema mismatches, missing columns, or data loading failures
        logger.error(f"An error occurred in TopN_customer_total_balance: {str(e)}", exc_info=True)
        return None

# To run and visualize it in your Databricks environment safely:
top_customers_df: Optional[DataFrame] = TopN_customer_total_balance(BASE_DIR, n=5)

if top_customers_df is not None:
    display(top_customers_df)
else:
    print("Failed to compute top customers due to an execution error.")






In [0]:
#TASK 2A
def greater_than_500(BASE_DIR: str) -> Optional[DataFrame]:
    ''' The aim of this function is to determine the number of transactions with an amount 
    greater than 500 within the last 30 days.
    
    The function will return a Spark DataFrame with six columns: 
    account_id, customer_id, first_name, last_name, amount and transaction_date.
    '''
    try:
        # 1. Fetch Master Data
        master_df = Master_Data(BASE_DIR)
        
        # 2. Get the maximum transaction date in the dataset
        max_date_row = master_df.select(F.max("transaction_date")).collect()
        max_date = max_date_row[0][0] if max_date_row else None
        
        if max_date is None:
            logger.warning("No data found in Master_File.")
            return None
            
        # 3. Calculate the Start Date (30 days prior to Max Date)
        past_30_days = F.date_add(F.lit(max_date), -30)
        
        # 4. Filter dataset and select required columns
        result_df = master_df.filter(
            (F.col("transaction_date") > past_30_days) & 
            (F.col("transaction_date") <= F.lit(max_date)) & 
            (F.col("transaction_type") == "Withdrawal") &
            (F.col("amount") > 500)
        ).select(
            "account_id", 
            "customer_id", 
            "first_name", 
            "last_name", 
            "amount", 
            "transaction_date"
        )
        
        # 5.  logs 
        logger.info(f"Max Date: {max_date}")
        logger.info(f"Filtering for transactions greater than $500 after: {max_date - datetime.timedelta(days=30) if 'datetime' in globals() else past_30_days}")
        logger.info(f"Total withdrawal records: {result_df.count()}")
        
        return result_df

    except Exception as e:
        logger.error(f"An error occurred while filtering transactions: {str(e)}")
        raise e


df_new_accounts = greater_than_500(BASE_DIR)
if df_new_accounts is not None:
    display(df_new_accounts)


In [0]:

#TASK 2B
def Deposit_Analysis_LBH(BASE_DIR: str) -> Optional[DataFrame]:
    """
    Calculates the total deposit amount per customer within the last 6 months 
    based on the most recent transaction date in the dataset.
    
    Args:
        BASE_DIR (str): The base directory path for the master data.
        
    Returns:
        Optional[DataFrame]: A Spark DataFrame containing customer_id and total_deposit_amount,
                             or None if no data is found.
    """
    try:
        # 1. Load Data
        master_file = Master_Data(BASE_DIR)
        
        # 2. Get the reference date (Max date in the dataset)
        max_date_row = master_file.select(F.max("transaction_date")).first()
        
        if not max_date_row or max_date_row[0] is None:
            logger.warning("No data found in Master_File.")
            return None
        
        max_date = max_date_row[0]
        logger.info(f"Most recent transaction date identified: {max_date}")
        
        # 3. Calculate the Start Date (6 months prior)
        start_date = F.add_months(F.lit(max_date), -6)
     
        # 4. Filter, Group, and Aggregate
        # Note: Changed F.count to F.sum to reflect "total deposit amount" as stated in docstring
        result_df = (
            master_file
            .filter(
                (F.col("transaction_date") >= start_date) &
                (F.col("transaction_date") <= F.lit(max_date)) &
                (F.col("transaction_type") == "Deposit")
            )
            .groupBy("customer_id")
            .agg(F.count("transaction_type").alias("total_deposit_amount"))
        )
        
        return result_df

    except Exception as e:
        logger.error(f"An error occurred during deposit analysis processing: {str(e)}")
        raise e
result_df = Deposit_Analysis_LBH(BASE_DIR)
display(result_df)

In [0]:
#TASK 2C
def calculate_running_balance() -> Optional[DataFrame]:
    """
     
    Calculates running balance based on transaction types.
    If the first installment/transaction is negative, withdrawals are permitted 
    to remain negative; otherwise, they are handled normally.
        
    Returns:
        DataFrame: The DataFrame with the calculated running balance.
    
    
    Business Rules Applied:
      - Deposits: Always added as positive attributes.
      - Withdrawals: Subtracted unless it is the very first transaction of the 
        account AND that first action was recorded as a negative number.
      - Payments: Negative values represent a refund (added); positive values 
        represent standard outbound debit flows (subtracted cleanly).
      - Transfers: Signed natural value passed straight through.
    """
    try:
        master_file = Master_Data(BASE_DIR)
        logger.info("Validating source DataFrame metrics (Serverless optimized)...")
        if not master_file.first():
            logger.warning("The input DataFrame holds no record segments. Returning None.")
            return None

        # 1. Define window tracking dimensions sequentially per account
        account_chronology_window = (
            Window.partitionBy("account_id")
            .orderBy("transaction_date", "transaction_id")
        )
        
        # Unbounded frame matrix to pull fixed anchor benchmarks globally per account partition
        global_lookup_window = (
            Window.partitionBy("account_id")
            .orderBy("transaction_date", "transaction_id")
            .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
        )

        logger.info("Step 1: Calculating transactional sequence indexes and opening benchmarks...")
        # Add sequence integers (1, 2, 3...) and find the absolute opening amount
        staged_df = master_file \
            .withColumn("tx_sequence", F.row_number().over(account_chronology_window)) \
            .withColumn("first_installment_amount", F.first("amount").over(global_lookup_window))

        logger.info("Step 2: Structuring defensively aligned operational ledger amounts...")
        # Transform structural raw amounts into accurate ledger math units
        df_with_signed_amounts = staged_df.withColumn(
            "calculated_amount",
            F.when(F.col("transaction_type") == "Deposit", F.abs(F.col("amount")))
            
            # Withdrawal Rule Matrix: Force subtract if it ISN'T row #1. 
            # If it IS row #1, permit negative passing only if first_installment_amount < 0.
            .when(F.col("transaction_type") == "Withdrawal", 
                F.when(F.col("tx_sequence") > 1, -F.abs(F.col("amount")))
                .otherwise(
                    F.when(F.col("first_installment_amount") < 0, F.col("amount"))
                    .otherwise(-F.abs(F.col("amount")))
                )
            )
            
            # Payment Matrix: Negative = Refund (+), Positive = Debit Outbound (-)
            .when(F.col("transaction_type") == "Payment", 
                F.when(F.col("amount") < 0, F.abs(F.col("amount")))
                .otherwise(-F.abs(F.col("amount")))
            )
            
            .when(F.col("transaction_type") == "Transfer", F.col("amount"))
            .otherwise(F.lit(0))
        )

        logger.info("Step 3: Compiling chronological cumulative balance streams...")
        # Define historical window tracking up to the active localized current row
        cumulative_window = account_chronology_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        
        final_report_df = df_with_signed_amounts.withColumn(
            "running_balance",
            F.sum("calculated_amount").over(cumulative_window)
        )
        
        # Drop temporary indexing metrics to return/ select only the final metrics
       # cleaned_report_df = final_report_df.drop("tx_sequence", "first_installment_amount", "calculated_amount")
        cleaned_report_df = final_report_df.select(
            "account_id", 
            "transaction_id",
            "transaction_date",
            "transaction_type", 
            "amount",
            "running_balance"
        ).orderBy("account_id", "transaction_date")
        
        logger.info("Running balance matrix aggregation pipeline completed successfully.")
        return cleaned_report_df

    except Exception as e:
        logger.error(f"Critical execution barrier encountered during ledger synthesis: {str(e)}")
        raise e

result = calculate_running_balance()
display(result)


In [0]:
#TASK3A
def Mean_Analysis():
    """
    Computes the mean transaction and balance amount for each account
    over the last 12 months.
    """
    # 1. Load Data
    try:
        logger.info("Loading master data file...")
        # 1. Load Data
        master_file = Master_Data(BASE_DIR)
        
        logger.info("Determining the date range for analysis...")
        # 2. Determine the date range
        # Get the latest date in the data as a Python date object
        max_date_row = master_file.select(F.max("transaction_date")).first()
        
        if not max_date_row or max_date_row[0] is None:
            logger.warning("No transaction data found in Master_File. Returning None.")
            return None # Handle empty DataFrame case
        
        max_date = max_date_row[0]
        logger.info(f"Latest transaction date identified: {max_date}")
        
        logger.info("Filtering data for the last 12 months...")
        # 3. Filter data to the last 12 months
        # We use add_months on the literal max_date to find our cutoff
        filtered_df = master_file.filter(
            F.col("transaction_date") >= F.add_months(F.lit(max_date), -12)
        )
        
        logger.info("Grouping and aggregating mean values...")
        # 4. Group by customer_id and calculate means
        mean_df = (
            filtered_df.groupBy("customer_id")
            .agg(
                F.mean("amount").alias("avg_transaction_amount"),
                F.mean("balance").alias("avg_balance")
            )
        )
        
        logger.info("Mean analysis calculation completed successfully.")
        return mean_df

    except Exception as e:
        logger.error(f"An error occurred during Mean_Analysis execution: {str(e)}")
        raise e
# RUN   
mean_df = Mean_Analysis()
if mean_df:
    display(mean_df)

In [0]:
#TASK-3B
def Customer_with_most_transactions() -> Optional[DataFrame]:
    """
    Computes the customer with the most transactions in the last 3 months.
    
        
    Returns:
        Optional[DataFrame]: A Spark DataFrame with 1 row containing the top customer,
                             or None if no data is found.
    """
    try:
        logger.info("Loading master data file...")
        # 1. Load Data
        master_file = Master_Data(BASE_DIR)
        
        logger.info("Determining the 3-month date window...")
        # Get the latest transaction date
        max_date_row = master_file.select(F.max("transaction_date")).first()
        
        if not max_date_row or max_date_row[0] is None:
            logger.warning("No transaction data found in Master_File. Returning None.")
            return None
        
        max_date = max_date_row[0]
        start_date = F.add_months(F.lit(max_date), -3)
        logger.info(f"Analysis window: {max_date} back to 3 months prior.")
        
        logger.info("Filtering, aggregating, and ranking transaction counts...")
        # 2. Filter, Group by customer info, Count, and extract the top record
        result_df = (
            master_file
            .filter(
                (F.col("transaction_date") > start_date) &
                (F.col("transaction_date") <= F.lit(max_date))
            )
            .groupBy("customer_id", "first_name", "last_name")
            .agg(F.count("transaction_id").alias("number_of_transactions"))
            .orderBy(F.desc("number_of_transactions"))
            .limit(1)
        )
        
        logger.info("Successfully identified the customer with the most transactions.")
        return result_df

    except Exception as e:
        logger.error(f"An error occurred during Customer_with_most_transactions execution: {str(e)}")
        raise e

most_transactions = Customer_with_most_transactions()
print(f"Customer with the most transactions in the last 3 months: ")
display(most_transactions)




In [0]:
#Task 4 
def Balance_Discrepancies() -> Optional[DataFrame]:
    """ 
    Computes the difference between the calculated balance and the reported balance.
    Serverless compliant (No RDD calls).
    
    Args:
        BASE_DIR (str): The base directory path for the master data.
        
    Returns:
        Optional[DataFrame]: A Spark DataFrame containing account IDs with mismatches,
                             or None if no source data is present.
    """
    try:
        logger.info("Loading master data file...")
        # 1. Load Data
        master_file = Master_Data(BASE_DIR)
        
        logger.info("Checking if DataFrame contains data (Serverless optimized)...")
        # SERVERLESS FIX: Replaced .rdd.isEmpty() with a DataFrame equivalent
        if not master_file.first():
            logger.warning("Master data is empty. Returning None.")
            return None

        logger.info("Step 1: Calculating the total sum of ledger transactions per account...")
        # Calculate the net transactional sum per account
        txn_sums = master_file.groupBy("account_id").agg(
            F.sum(
                F.when(F.col("transaction_type") == "Deposit", F.abs(F.col("amount")))
                .when(F.col("transaction_type") == "Withdrawal", -F.abs(F.col("amount")))
                .when(F.col("transaction_type") == "Payment", 
                    F.when(F.col("amount") < 0, F.abs(F.col("amount")))  # Negative Payment = Refund (Credit)
                    .otherwise(-F.abs(F.col("amount")))                  # Positive Payment = Debit (Outbound)
                )
                .when(F.col("transaction_type") == "Transfer", F.col("amount"))
                .otherwise(F.lit(0))
            ).alias("calculated_balance")
        )

        logger.info("Step 2: Extracting the datas...")
        # Grab the max/latest balance snapshot
        account_balances = master_file.groupBy("account_id").agg(
            F.max("balance").alias("reported_balance")
        )

        logger.info("Step 3:  join data frame together...")
        discrepancies = txn_sums.join(account_balances, "account_id", "inner")

        logger.info("Step 4: Filtering out unaffected records...")
        result_df = discrepancies.filter(
            F.round(F.col("reported_balance"), 2) != F.round(F.col("calculated_balance"), 2)
        ) 
        
        logger.info("Reconciliation analysis process complete.")
        return result_df

    except Exception as e:
        logger.error(f"An error occurred during Balance_Discrepancies processing: {str(e)}")
        raise e
result = Balance_Discrepancies()
display(result)





In [0]:
#Task 4B
def Missing_Or_Incomplete_Records() -> Optional[DataFrame]:
    try:
            logger.info("Loading dictionary of source data CSVs...")
            # Load the raw files dictionary
            dataframes: Dict[str, DataFrame] = load_csvs_to_dict(BASE_DIR)
        
                
            customers_df = dataframes["customers"]
            
            logger.info("Checking if customer dataset contains data (Serverless optimized)...")
            # Serverless-compliant verification check
            if not customers_df.first():
                logger.warning("Customers DataFrame is completely empty. Returning None.")
                return None

            logger.info("Step 1: Generating missing logic array for profile fields...")
            # Standard structural profile columns to assess
            check_cols = ["first_name", "last_name", "date_of_birth", "address"]
            
            # Create expressions for standard null/empty evaluation checks
            missing_logic = [
                F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), F.lit(c)).otherwise(None) 
                for c in check_cols
            ]
            
            logger.info("Step 2: Adding domain validation checks for zip code structures...")
            # Specific business logic rules for ZIP code entries (Length < 5)
            zip_logic = F.when(
                F.col("zip").isNull() | 
                (F.trim(F.col("zip").cast("string")) == "") | 
                (F.length(F.trim(F.col("zip").cast("string"))) < 5), 
                F.lit("zip")
            ).otherwise(None)
            
            missing_logic.append(zip_logic)
            
            logger.info("Step 3: Compiling tracking strings and executing filtering matrices...")
            # Build out the final anomaly tracking data structure
            result_df = (
                customers_df
                .withColumn("missing_fields", F.concat_ws(", ", *missing_logic))
                # Filter rows where missing_fields holds structural strings
                .filter(F.col("missing_fields") != "")
                .select(
                    "customer_id", 
                    "first_name", 
                    "last_name", 
                    "missing_fields"
                )
                .dropDuplicates(["customer_id"])
            )
            
            logger.info("Data profiling complete. Returning anomaly dataset.")
            return result_df

    except Exception as e:
        logger.error(f"An error occurred while tracking incomplete profiles: {str(e)}")
        raise e

# Run and display
incomplete_customers = Missing_Or_Incomplete_Records()
display(incomplete_customers)

In [0]:
#Task 4C 

def Identify_Duplicate_Accounts()-> Optional[DataFrame]:
    try:
        dataframes: Dict[str, DataFrame] = load_csvs_to_dict(BASE_DIR)
        accounts_df = dataframes["accounts"]
        logger.info("Verifying input accounts DataFrame (Serverless optimized)...")
        # Serverless-compliant check to ensure the DataFrame is not empty
        if not accounts_df.first():
            logger.warning("The input accounts DataFrame is empty. Returning None.")
            return None

        logger.info("Executing GROUP BY and HAVING COUNT(*) > 1 logic...")
        # PySpark equivalent to your SQL query
        result_df = (
            accounts_df
            .groupBy("customer_id", "account_type")
            .agg(F.count("*").alias("number_of_duplicates"))
            .filter(F.col("number_of_duplicates") > 1)
        )
        
        logger.info("Successfully identified duplicate accounts.")
        return result_df

    except Exception as e:
        logger.error(f"An error occurred while processing duplicate accounts: {str(e)}")
        raise e
    
# Execute and display in Databricks
duplicate_report = Identify_Duplicate_Accounts()
display(duplicate_report)

In [0]:
#Task 4D 
def Invalid_Unrecognised_Transaction_Type() -> Optional[DataFrame]: 
    """
    The purpose of the function is to identify customers with invalid or 
    unrecognised transaction types (i.e., other than Deposit, Withdrawal, Transfer, Payment).
    
    Returns a Spark DataFrame with three columns: transaction_id, account_id, and transaction_type.
    """
    try:
        # Load the master data (assuming Master_Data returns a Spark DataFrame)
        master_file: DataFrame = Master_Data(BASE_DIR) 
        
        # Define the list of valid transaction types
        valid_types = ["Deposit", "Withdrawal", "Payment", "Transfer"]

        # Filter for transactions NOT in the valid types list
        invalid_txns: DataFrame = master_file.filter(~F.col("transaction_type").isin(valid_types))

        # Select the required columns for the output
        output: DataFrame = invalid_txns.select("transaction_id", "account_id", "transaction_type")

        # Performance Optimization: Avoid calling .count() on a cluster just to check for emptiness.
        # .isEmpty() is significantly faster as it evaluates only the first record.
        if not output.isEmpty():
            logger.info("Invalid or unrecognised transactions detected.")
            return output
        else:
            logger.info("No invalid transactions found.")
            return None

    except Exception as e:
        # Catch issues like missing columns or file path execution errors
        logger.error(f"An error occurred in Invalid_Unrecognised_Transaction_Type: {str(e)}", exc_info=True)
        return None

# To run and visualize it in your Databricks environment safely:
Df: Optional[DataFrame] = Invalid_Unrecognised_Transaction_Type()

if Df is not None:
    display(Df)
else:
    print("No invalid transactions to display.")



In [0]:
#TASK 4E 

def Identify_Non_Credit_Accounts_with_Negative_Balance(): 
    try:
        logger.info("Loading master data file...")
        # 1. Load Data
        master_file = Master_Data(BASE_DIR)
        
        logger.info("Checking if dataset contains data (Serverless optimized)...")
        # Serverless-compliant verification check
        if not master_file.first():
            logger.warning("Master data file is completely empty. Returning None.")
            return None

        logger.info("Step 1: Filtering out Credit accounts and isolating negative balances...")
        # 2. Apply filtering logic
        negative_balance_accounts = master_file.filter(
            (F.col("account_type") != "Credit") & 
            (F.col("balance") < 0)
        )
        
        logger.info("Step 2: Projecting target attributes and dropping duplicate account rows...")
        # 3. Select target columns and drop duplicates based on the account identifier
        result_df = negative_balance_accounts.select("customer_id", "account_id", "account_type", "balance")
        result_df = result_df.dropDuplicates(["account_id"])
            
        
        
        logger.info("Negative balance non-credit account identification complete.")
        return result_df
    
    except Exception as e:
        logger.error(f"An error occurred while identifying negative balance accounts: {str(e)}")
        raise e
    return
non_cred = Identify_Non_Credit_Accounts_with_Negative_Balance()
display(non_cred)